# Does a Vision-Language Model Understand Stability?

This notebook generates simple 2D block-stacking scenes, computes a physics-inspired stability label,
and optionally compares that label with the prediction of a vision-language model.

## Learning goals

- Generate simple physical scenes
- Compute an interpretable stability label from geometry
- Query a multimodal model about the same scene
- Compare correctness vs explanation quality
- Analyze failure modes near the stability boundary

## Notebook modes

**Mode A — no API required**

You can run the notebook end-to-end, generate scenes, compute ground truth, and inspect examples.

**Mode B — optional VLM evaluation**

If you have access to a multimodal API, you can enable the VLM section and compare model predictions with ground truth.

In [ ]:
import math
import random
import json
import os
from dataclasses import dataclass
from typing import List, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## Physical model

We use a simplified 2D static-equilibrium rule.

For a sub-stack resting on the block below it, the horizontal projection of the sub-stack center of mass
must lie inside the support interval of the supporting block.

This is not full rigid-body simulation, but it is interpretable and useful for a first experiment.

In [ ]:
@dataclass
class Block:
    x: float          # center x
    y: float          # bottom y
    w: float          # width
    h: float          # height
    mass: float       # mass
    name: str = "block"

    @property
    def left(self) -> float:
        return self.x - self.w / 2

    @property
    def right(self) -> float:
        return self.x + self.w / 2

    @property
    def com_x(self) -> float:
        return self.x

    @property
    def com_y(self) -> float:
        return self.y + self.h / 2

In [ ]:
def combined_com_x(blocks: List[Block]) -> float:
    total_mass = sum(b.mass for b in blocks)
    if total_mass <= 0:
        raise ValueError("Total mass must be positive.")
    return sum(b.mass * b.com_x for b in blocks) / total_mass


def is_stack_stable(blocks: List[Block], eps: float = 1e-9) -> bool:
    '''
    Blocks must be ordered from bottom to top.
    For each interface, the COM of the sub-stack above must lie within the support
    interval of the supporting block.
    '''
    if len(blocks) <= 1:
        return True

    for i in range(len(blocks) - 1):
        support = blocks[i]
        upper_substack = blocks[i + 1:]
        com_x = combined_com_x(upper_substack)
        if com_x < support.left - eps or com_x > support.right + eps:
            return False
    return True


def stability_margin(blocks: List[Block]) -> float:
    '''
    Minimum signed margin across interfaces.
    Positive => stable, negative => unstable.
    '''
    if len(blocks) <= 1:
        return float("inf")

    margins = []
    for i in range(len(blocks) - 1):
        support = blocks[i]
        upper_substack = blocks[i + 1:]
        com_x = combined_com_x(upper_substack)
        margins.append(min(com_x - support.left, support.right - com_x))
    return min(margins)

In [ ]:
def generate_stack(
    n_blocks: int = 3,
    base_y: float = 0.0,
    width_range=(1.0, 2.2),
    height_range=(0.4, 0.9),
    mass_range=(0.5, 3.0),
    offset_scale=0.6,
    seed: Optional[int] = None,
) -> List[Block]:
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    blocks = []

    w = random.uniform(*width_range)
    h = random.uniform(*height_range)
    m = random.uniform(*mass_range)
    blocks.append(Block(x=0.0, y=base_y, w=w, h=h, mass=m, name="base"))

    for i in range(1, n_blocks):
        prev = blocks[-1]
        w = random.uniform(*width_range)
        h = random.uniform(*height_range)
        m = random.uniform(*mass_range)

        offset = random.uniform(-offset_scale, offset_scale) * prev.w
        x = prev.x + offset
        y = prev.y + prev.h

        blocks.append(Block(x=x, y=y, w=w, h=h, mass=m, name=f"block_{i}"))

    return blocks


def generate_labeled_scene(
    target_label: Optional[int] = None,
    max_tries: int = 1000,
    n_blocks_range=(2, 5),
    seed: Optional[int] = None,
):
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    for _ in range(max_tries):
        n_blocks = random.randint(*n_blocks_range)
        blocks = generate_stack(n_blocks=n_blocks)
        stable = int(is_stack_stable(blocks))
        if target_label is None or stable == target_label:
            return blocks, stable, stability_margin(blocks)

    raise RuntimeError("Failed to generate a scene with the requested label.")

In [ ]:
def draw_scene(
    blocks: List[Block],
    show_com: bool = True,
    show_support_lines: bool = False,
    title: Optional[str] = None,
    figsize=(6, 4),
):
    fig, ax = plt.subplots(figsize=figsize)

    for i, b in enumerate(blocks):
        rect = Rectangle((b.left, b.y), b.w, b.h, fill=True, linewidth=2)
        ax.add_patch(rect)
        ax.text(b.x, b.y + b.h / 2, f"{i}", ha="center", va="center", fontsize=10)

        if show_support_lines:
            ax.plot([b.left, b.right], [b.y, b.y], linestyle="--", linewidth=1)

    if show_com:
        for i in range(len(blocks) - 1):
            upper = blocks[i + 1:]
            cx = combined_com_x(upper)
            y = blocks[i].y + blocks[i].h + 0.05
            ax.plot(cx, y, marker="o")
            ax.text(cx, y + 0.05, f"COM↑[{i+1}:]", ha="center", fontsize=8)

    stable = is_stack_stable(blocks)
    margin = stability_margin(blocks)

    ax.set_aspect("equal")
    ax.set_xlim(-4, 4)
    ax.set_ylim(0, sum(b.h for b in blocks) + 1)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.grid(True, alpha=0.2)

    if title is None:
        title = f"Stable={stable}, margin={margin:.2f}"
    ax.set_title(title)

    plt.show()

## Quick demo

In [ ]:
blocks, stable, margin = generate_labeled_scene(target_label=1, seed=42)
draw_scene(blocks, show_com=True, show_support_lines=True)
print("Stable:", stable, "Margin:", margin)

## Building a small synthetic dataset

In [ ]:
def build_dataset(n_samples: int = 40, seed: int = 0) -> pd.DataFrame:
    random.seed(seed)
    np.random.seed(seed)

    rows = []
    for i in range(n_samples):
        target_label = i % 2
        blocks, stable, margin = generate_labeled_scene(target_label=target_label)

        rows.append({
            "id": i,
            "blocks": blocks,
            "stable": stable,
            "margin": margin,
            "difficulty": "hard" if abs(margin) < 0.2 else "easy",
        })

    return pd.DataFrame(rows)

In [ ]:
df = build_dataset(n_samples=30, seed=1)
df.head()

In [ ]:
for idx in range(4):
    row = df.iloc[idx]
    draw_scene(
        row["blocks"],
        title=f"id={row['id']} stable={row['stable']} difficulty={row['difficulty']}"
    )

## Saving scene images

This is needed only for optional VLM querying.

In [ ]:
os.makedirs("scene_images", exist_ok=True)

def save_scene_image(blocks: List[Block], filepath: str, title: Optional[str] = None):
    fig, ax = plt.subplots(figsize=(6, 4))

    for b in blocks:
        rect = Rectangle((b.left, b.y), b.w, b.h, fill=False, linewidth=2)
        ax.add_patch(rect)

    ax.set_aspect("equal")
    ax.set_xlim(-4, 4)
    ax.set_ylim(0, sum(b.h for b in blocks) + 1)
    ax.axis("off")

    if title is not None:
        ax.set_title(title)

    plt.tight_layout()
    plt.savefig(filepath, dpi=150, bbox_inches="tight")
    plt.close(fig)

## Optional VLM evaluation

The notebook is fully usable without this section.

If you have a multimodal API, replace the placeholder function below with your actual API call.

In [ ]:
def make_prompt() -> str:
    return (
        "You are judging 2D block stability.\n"
        "Look at the stack of rectangles.\n"
        "Answer with exactly three lines:\n"
        "label: stable OR unstable\n"
        "reason: one short sentence\n"
        "fix: one short sentence\n"
        "Judge physical stability, not visual neatness."
    )

import re
from PIL import Image

import re

def parse_three_line_output(text: str) -> dict:
    text = text.strip()
    lower = text.lower()

    label = ""
    reason = ""
    fix = ""

    # Case 1: structured output
    m = re.search(r"label\s*:\s*(stable|unstable)", lower)
    if m:
        label = m.group(1)

    m = re.search(r"reason\s*:\s*(.+)", text, flags=re.IGNORECASE)
    if m:
        reason = m.group(1).strip()

    m = re.search(r"fix\s*:\s*(.+)", text, flags=re.IGNORECASE)
    if m:
        fix = m.group(1).strip()

    # Case 2: bare one-word answer
    if label == "":
        if re.fullmatch(r"\s*stable\.?\s*", lower):
            label = "stable"
        elif re.fullmatch(r"\s*unstable\.?\s*", lower):
            label = "unstable"

    # Case 3: label appears anywhere
    if label == "":
        if "unstable" in lower:
            label = "unstable"
        elif "stable" in lower:
            label = "stable"

    # If only a bare answer was returned, keep it as reason too
    if reason == "":
        reason = text

    return {
        "label": label,
        "reason": reason,
        "fix": fix,
        "raw_output": text,
    }

def query_vlm_hf(image_path: str, prompt: str, max_new_tokens: int = 16) -> dict:
    image = Image.open(image_path).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    outputs = hf_pipe(
        text=messages,
        max_new_tokens=max_new_tokens,
        return_full_text=False,
    )

    generated = outputs[0]["generated_text"]

    if isinstance(generated, list):
        parts = []
        for item in generated:
            if isinstance(item, dict) and item.get("type") == "text":
                parts.append(item.get("text", ""))
        generated_text = "".join(parts).strip()
    else:
        generated_text = str(generated).strip()

    return parse_three_line_output(generated_text)

def make_prompt_label_only() -> str:
    return (
        "Look at this 2D stack of rectangular blocks. "
        "Is it physically stable or unstable? "
        "Answer with exactly one word: stable or unstable."
    )


### Example pseudocode for an API-based multimodal query

Adapt this to your provider and credentials. Keep this cell commented until you are ready to use it.

In [ ]:
# Example sketch only — not executable as-is.
#
# import base64
# from openai import OpenAI
#
# client = OpenAI()
#
# def encode_image(path: str) -> str:
#     with open(path, "rb") as f:
#         return base64.b64encode(f.read()).decode("utf-8")
#
# def query_vlm(image_path: str, prompt: str) -> dict:
#     image_b64 = encode_image(image_path)
#     response = client.responses.create(
#         model="gpt-4.1",
#         input=[
#             {
#                 "role": "user",
#                 "content": [
#                     {"type": "input_text", "text": prompt},
#                     {
#                         "type": "input_image",
#                         "image_url": f"data:image/png;base64,{image_b64}",
#                     },
#                 ],
#             }
#         ],
#     )
#     text = response.output_text
#     return json.loads(text)

In [ ]:
def evaluate_with_vlm(df: pd.DataFrame) -> pd.DataFrame:
    preds = []

    for _, row in df.iterrows():
        image_path = f"scene_images/scene_{row['id']}.png"
        save_scene_image(row["blocks"], image_path)

        result = query_vlm_hf(
            image_path,
            make_prompt_label_only(),
            max_new_tokens=8
        )

        pred_label_text = result.get("label", "").strip().lower()
        pred_stable = 1 if pred_label_text == "stable" else 0

        preds.append({
            "id": row["id"],
            "pred_label_text": pred_label_text,
            "pred_stable": pred_stable,
            "reason": result.get("reason", ""),
            "fix": result.get("fix", ""),
        })

    return df.merge(pd.DataFrame(preds), on="id")

## Manual evaluation mode

If no API is available, students can still inspect images and fill predictions manually.

In [ ]:
def build_manual_template(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in df.iterrows():
        image_path = f"scene_images/scene_{row['id']}.png"
        save_scene_image(row["blocks"], image_path)
        rows.append({
            "id": row["id"],
            "image_path": image_path,
            "true_label": row["stable"],
            "student_pred": None,
            "student_reason": "",
        })
    return pd.DataFrame(rows)

manual_df = build_manual_template(df)
manual_df.head()

## Evaluation metrics

In [ ]:
def evaluate_predictions(df_pred: pd.DataFrame, pred_col: str = "pred_stable", true_col: str = "stable"):
    y_true = df_pred[true_col].astype(int).tolist()
    y_pred = df_pred[pred_col].astype(int).tolist()

    print("Accuracy:", accuracy_score(y_true, y_pred))
    print()
    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred))
    print()
    print("Classification report:")
    print(classification_report(y_true, y_pred, target_names=["unstable", "stable"]))

In [ ]:
def evaluate_by_difficulty(df_pred: pd.DataFrame, pred_col: str = "pred_stable", true_col: str = "stable"):
    for difficulty in ["easy", "hard"]:
        sub = df_pred[df_pred["difficulty"] == difficulty]
        if len(sub) == 0:
            continue
        acc = accuracy_score(sub[true_col], sub[pred_col])
        print(f"{difficulty}: n={len(sub)}, accuracy={acc:.3f}")

## Failure analysis

In [ ]:
def inspect_examples(df_pred: pd.DataFrame, correct: Optional[bool] = None, n: int = 5,
                     pred_col: str = "pred_stable", true_col: str = "stable"):
    if correct is None:
        sub = df_pred
    else:
        sub = df_pred[(df_pred[true_col] == df_pred[pred_col]) == correct]

    sub = sub.head(n)

    for _, row in sub.iterrows():
        print("=" * 80)
        print(f"id={row['id']}  true={row[true_col]}  pred={row[pred_col]}  difficulty={row['difficulty']}")
        print(f"margin={row['margin']:.3f}")
        if "reason" in row:
            print("reason:", row.get("reason", ""))
        if "fix" in row:
            print("fix:", row.get("fix", ""))
        draw_scene(row["blocks"], title=f"id={row['id']}")

## Suggested student questions

1. Is the model better on clearly stable or clearly unstable scenes than on borderline scenes?
2. Are the explanations physically meaningful, or only plausible-sounding?
3. Does the model identify the correct support failure?
4. When the model is wrong, is the mistake geometric, semantic, or linguistic?
5. Does the model suggest a reasonable stabilizing action?

## Optional extensions

### A. Hidden mass
Keep geometry fixed but change masses. This tests whether the model relies only on visible geometry.

### B. Damage risk
Add fragile-support labels and ask whether damage is likely.

### C. Affordance + physics
Ask:
1. where should the robot grasp?
2. is that grasp safe?

This connects the notebook directly to affordance reasoning and VLA discussion.

## Local Hugging Face VLM integration

This section replaces the API dependency with a local Hugging Face model.

### Recommended defaults

- `HuggingFaceTB/SmolVLM-256M-Instruct` is the safest Colab starting point.
- If you have a stronger GPU, you can switch to a larger local VLM later.

The code below uses the `image-text-to-text` pipeline, which is the current Hugging Face entry point for multimodal chat-style inference.

In [ ]:
# Install only if needed in Colab.
# Uncomment the next line in a fresh runtime.
# !pip -q install transformers accelerate pillow

In [ ]:
from transformers import pipeline
from PIL import Image
import torch

### Model loading

Start with a small model first. This keeps the lab reliable on standard Colab GPUs.

In [ ]:
HF_VLM_MODEL = "HuggingFaceTB/SmolVLM-256M-Instruct"

# Notes:
# - device_map="auto" lets Transformers choose GPU/CPU placement.
# - torch_dtype="auto" is the safest portable choice across Colab runtimes.

hf_pipe = pipeline(
    task="image-text-to-text",
    model=HF_VLM_MODEL,
    device_map="auto",
    torch_dtype="auto",
)
print("Loaded:", HF_VLM_MODEL)

### Local query function

The prompt format follows the multimodal chat template style expected by modern Hugging Face VLMs.

In [ ]:
def query_vlm_hf(image_path: str, prompt: str, max_new_tokens: int = 120) -> dict:
    image = Image.open(image_path).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    outputs = hf_pipe(
        text=messages,
        max_new_tokens=max_new_tokens,
        return_full_text=False,
    )

    generated = outputs[0]["generated_text"]

    # Try strict JSON first; if parsing fails, return raw text.
    try:
        if isinstance(generated, list):
            # Some models return structured chat content
            text_chunks = []
            for item in generated:
                if isinstance(item, dict) and item.get("type") == "text":
                    text_chunks.append(item.get("text", ""))
            generated_text = "".join(text_chunks).strip()
        else:
            generated_text = str(generated).strip()

        return json.loads(generated_text)
    except Exception:
        return {
            "label": "",
            "reason": str(generated),
            "fix": "",
            "raw_output": str(generated),
        }

### Switch the notebook to local inference

In [ ]:
# Replace the generic query function with the local Hugging Face version.
query_vlm = query_vlm_hf

### Single-example smoke test

In [ ]:
import re
from PIL import Image

def parse_three_line_output(text: str) -> dict:
    text = text.strip()
    lower = text.lower()

    label = ""
    reason = ""
    fix = ""

    # Structured output
    m = re.search(r"label\s*:\s*(stable|unstable)", lower)
    if m:
        label = m.group(1)

    m = re.search(r"reason\s*:\s*(.+)", text, flags=re.IGNORECASE)
    if m:
        reason = m.group(1).strip()

    m = re.search(r"fix\s*:\s*(.+)", text, flags=re.IGNORECASE)
    if m:
        fix = m.group(1).strip()

    # Bare one-word answer
    if label == "":
        if re.fullmatch(r"\s*stable\.?\s*", lower):
            label = "stable"
        elif re.fullmatch(r"\s*unstable\.?\s*", lower):
            label = "unstable"

    # Fallback: label appears anywhere
    if label == "":
        if "unstable" in lower:
            label = "unstable"
        elif "stable" in lower:
            label = "stable"

    if reason == "":
        reason = text

    return {
        "label": label,
        "reason": reason,
        "fix": fix,
        "raw_output": text,
    }


def query_vlm_hf(image_path: str, prompt: str, max_new_tokens: int = 8) -> dict:
    image = Image.open(image_path).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    outputs = hf_pipe(
        text=messages,
        max_new_tokens=max_new_tokens,
        return_full_text=False,
    )

    generated = outputs[0]["generated_text"]

    if isinstance(generated, list):
        parts = []
        for item in generated:
            if isinstance(item, dict) and item.get("type") == "text":
                parts.append(item.get("text", ""))
        generated_text = "".join(parts).strip()
    else:
        generated_text = str(generated).strip()

    print("RAW MODEL OUTPUT:", repr(generated_text))
    parsed = parse_three_line_output(generated_text)
    print("PARSED OUTPUT:", parsed)
    return parsed

In [ ]:
print(parse_three_line_output(" Unstable."))
print(query_vlm_hf(test_image_path, simple_prompt, max_new_tokens=8))

In [ ]:
row = df.iloc[0]
test_image_path = f"scene_images/scene_{row['id']}_hf_test.png"
save_scene_image(row["blocks"], test_image_path)

simple_prompt = make_prompt_label_only()
result = query_vlm_hf(test_image_path, simple_prompt, max_new_tokens=8)
print(result)
draw_scene(row["blocks"], title=f"id={row['id']} stable={row['stable']}")

In [ ]:
df_pred = evaluate_with_vlm(df)
df_pred[["id", "stable", "pred_stable", "pred_label_text", "difficulty"]].head(10)

In [ ]:
for idx in [0, 1, 2]:
    row = df.iloc[idx]
    test_image_path = f"scene_images/debug_{row['id']}.png"
    save_scene_image(row["blocks"], test_image_path)

    print("=" * 60)
    print("ID:", row["id"])
    print("GROUND TRUTH stable:", row["stable"])
    draw_scene(row["blocks"], title=f"id={row['id']} stable={row['stable']}")

    result = query_vlm_hf(test_image_path, make_prompt_label_only(), max_new_tokens=8)
    print("MODEL:", result)

### Batch evaluation with the local Hugging Face model

In [ ]:
# This may take a few minutes depending on Colab GPU/CPU speed.
# df_pred = evaluate_with_vlm(df)
# df_pred.head()

### Evaluation

In [ ]:
# Uncomment after running batch evaluation
# evaluate_predictions(df_pred)
# evaluate_by_difficulty(df_pred)

### Practical note

If the smallest local model is too weak on the task, keep the exact same notebook structure and try a larger image-text-to-text model in the same `pipeline(...)` call.